# 032-optimised — GM record selection, AvgSA([0, 3]), second ("optimised") selection — Stage 1 (compute)

A **second, complete** record selection for every `(site, iml)` stripe, run alongside the original
(nb `032`) so the two record sets can be compared. Same engine (`build_final_ensembles`), same
GCIM targets, same selection context; only the round scheme and the database filter differ.

| round | select with | optimise with | which stripes are optimised |
|---|---|---|---|
| 1 | bounds on **M and Vs30, not distance** | same | **every** stripe (`force_optimisation=True`), including ones that pass straight after selection |
| 2 | **no bounds**; only stripes for which round 1 could not form a record set at all are reselected | no bounds | every stripe still failing, including the newly reselected ones even if they already pass. A round-1 set that failed restarts from its round-1 *selection* |
| 3 | no reselection | no bounds, best of 5 shuffled database orderings | stripes still failing; keeps the better of {new, old} |

A stripe that passes after any round is never touched again. The scheme is
`SELECTION_CONFIG_OPTIMISED` in `setup_AvgSA03_gm_selection.py`; the engine
(`build_final_ensembles`) is the same as the original selection's.

**Excluded records — where the filtering happens.** The records in
`setup_AvgSA03_gm_selection.UNAVAILABLE_RECORDS` (the NGA-Sub ONA network, and NGA-Sub RSN
6003695 / 6003748 / 6003796, whose waveforms are ~3–4.5e4 × their flatfile SA) are removed from
the database **once, in nb 032-optimised, cell "Set up + filter the database"**, by
`gm_db_for_selection_optimised(gm_db)`. That is the only place the filtering happens; every
later step receives only the filtered frame. The file on disk is never edited, and surviving rows
keep their original index labels. The exclusion is recorded in the provenance: the
`db_exclusions` entry of every stripe manifest, of the stage caches and of
`AvgSA_03_final_ensembles.pickle.manifest.json`.

**Kept apart from the original selection.** Everything this notebook writes goes to
`cfg["proc_data"]["gm_selection_optimised"]` (`data_processed/06_gm_selection/AvgSA_03_optimised/`)
and `cfg["results"]["AvgSA_03_record_selection_optimised"]`
(`results/04_record_selection/AvgSA_03_optimised/`), under the same file names as the original.
The original selection's files are only ever read, never written.

**Incremental**, exactly like nb `032`: `find_stale_stripes` (against the optimised result folder
and `SELECTION_CONFIG_OPTIMISED`) picks the stripes that are missing or stale, only their disagg
shards are loaded, and the batch is merged into `AvgSA_03_final_ensembles.pickle` **in the
optimised folder**, which stays the COMPLETE `(site, iml)` set. On the first run every stripe is
to (re)select. Set `FORCE_RECOMPUTE = True` to reselect every wanted stripe.

**Upstream**:

| Input | Source |
|---|---|
| GCIM target distributions (complete set) | `gcim_dist_AvgSA_03.pickle` (nb `031`) — **shared with the original**; the targets do not depend on the bounds, so nb 031 is not rerun |
| Combined selection GM database | `cfg["proc_data"]["gm_database"]`, filtered in memory (see above) |
| IML-based disagg shards + poe stats | `cfg["proc_data"]["AvgSA_03_disagg_data_shards"]`, `cfg["proc_data"]["AvgSA_03_disagg_stats_gm_selection"]` |
| Site model | `cfg["hazard_models"]["eshm20_wp1_site_model"]` |
| GMM logic tree + correlation flatfiles | loaded inside `setup_AvgSA03_gcim_gm_selection()` |

**Downstream** — `033-gm_selection_AvgSA_03_stage2_postprocess-optimised.ipynb`.

**Run order** — run top to bottom, then `033-…-optimised`. The first run selects all stripes
(likely over an hour; every stripe is optimised in round 1).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle
import numpy as np
import pandas as pd

from phd_project.config.config import load_config
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    build_final_ensembles,
    stripe_input_fingerprint,
    find_stale_stripes,
)
# NOTE: SELECTION_CONFIG_OPTIMISED, NOT the original SELECTION_CONFIG. Mixing them up
# would fingerprint the stripes against the wrong scheme.
from phd_project.scripts.WP1_ground_motion_set.setup_AvgSA03_gm_selection import (
    setup_AvgSA03_gcim_gm_selection,
    wanted_stripe_keys,
    SELECTION_CONFIG_OPTIMISED,
    stripe_source_fps,
    gm_db_for_selection_optimised,
)


cfg = load_config()

In [ ]:
# All OUTPUTS of this notebook live in the optimised folder; the original selection's
# folder (cfg["proc_data"]["gm_selection"]) is never written to.
GMS = cfg["proc_data"]["gm_selection_optimised"]
GMS.mkdir(parents=True, exist_ok=True)

# canonical Stage-1 output (merged each run to stay the COMPLETE (site, iml) set).
# Same file name as the original, in the optimised folder, so nb 040-optimised's
# summarise_availability() finds it unchanged.
final_ensembles_fp = GMS / "AvgSA_03_final_ensembles.pickle"

# intermediate stage caches, one per round (provenance-guarded; per-batch scratch).
# Named {im}_rd{ii}_{selection|optimisation}.pickle, in the optimised folder.
IM = "AvgSA_03"
n_rounds = len(SELECTION_CONFIG_OPTIMISED["round_unbounded"])
stage_fps = {
    "select":   [GMS / f"{IM}_rd{ii}_selection.pickle"    for ii in range(1, n_rounds + 1)],
    "optimise": [GMS / f"{IM}_rd{ii}_optimisation.pickle" for ii in range(1, n_rounds + 1)],
}

# Inputs used for provenance fingerprinting (hashed by their file bytes). These stage /
# final artifacts cover a whole batch, so the disagg entry is the digest of the ENTIRE
# shard set (disagg_shard_dir -> disagg_shards_digest), not any one site's shard — see
# gm_selection._disagg_fingerprint_input. The GCIM file is the ORIGINAL nb 031 output
# (read only): the targets do not depend on the bounds.
gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_03.pickle"
source_fps = {
    "gm_db_file":        cfg["proc_data"]["gm_database"],
    "gcim_file":         gcim_dist_fp,
    "disagg_shard_dir":  cfg["proc_data"]["AvgSA_03_disagg_data_shards"],
    "disagg_stats_file": cfg["proc_data"]["AvgSA_03_disagg_stats_gm_selection"],
    "site_model_file":   cfg["hazard_models"]["eshm20_wp1_site_model"],
}

# The selection scheme (rounds, bounds, shuffle, seeds, rng_seed, db_exclusions) lives in
# SELECTION_CONFIG_OPTIMISED (setup_AvgSA03_gm_selection.py) so 032-/033-optimised
# fingerprint each stripe identically. See that constant for the round-by-round description.
rng_seed = SELECTION_CONFIG_OPTIMISED["rng_seed"]

# Escape hatch: force-reselect ALL wanted stripes, ignoring the per-stripe manifests
# (and bypassing every stage's staleness guard). Leave False for normal incremental
# runs: only the stale/new (site, iml) are (re)selected.
FORCE_RECOMPUTE = False

In [ ]:
# --- Set up + filter the database ---------------------------------------------------
# Set up the selection context WITHOUT loading any disagg data (sites=()) — the stale
# check needs only the context and the index-derived key set. The shards for the stale
# sites are loaded in the compute cell, once we know which sites those are.
_, disagg_stats, site_model, basic_selection_ctx, gm_db_full = setup_AvgSA03_gcim_gm_selection(sites=())

# THE database filter of the optimised selection (the only place it happens): drop every
# record in UNAVAILABLE_RECORDS, in memory, keeping the original index labels. From here
# on ONLY `gm_db` (filtered) is used; `gm_db_full` is kept for the check below.
gm_db = gm_db_for_selection_optimised(gm_db_full)
excluded_idx = gm_db_full.index.difference(gm_db.index)
if len(excluded_idx) == 0:
    raise RuntimeError("The filter dropped no rows: UNAVAILABLE_RECORDS matches nothing in "
                       "the database, so the exclusion is not in effect.")

# Incremental cache: (re)select only stripes missing or stale in the OPTIMISED result
# folder. The fingerprint uses SELECTION_CONFIG_OPTIMISED, which carries db_exclusions,
# so the exclusion is part of every stripe's manifest.
RESULT_FOLDER = cfg["results"]["AvgSA_03_record_selection_optimised"]
RESULT_FOLDER.mkdir(parents=True, exist_ok=True)
source_fps_stripe = stripe_source_fps()
wanted = wanted_stripe_keys()
fp_fn = lambda s, i: stripe_input_fingerprint(
    s, i, source_fps_stripe, basic_selection_ctx, SELECTION_CONFIG_OPTIMISED, disagg_stats)
to_compute, valid = find_stale_stripes(wanted, RESULT_FOLDER, fp_fn)
print(f"{len(wanted)} wanted (site, iml): {len(valid)} already valid, "
      f"{len(to_compute)} to (re)select.")

# load the (complete) gcim distributions written by nb 031 (shared with the original)
if gcim_dist_fp.is_file():
    with open(gcim_dist_fp, "rb") as file:
        gcim_dists = pickle.load(file)
    print(f"Loaded gcim for {len(gcim_dists)} (site, iml).")
else:
    gcim_dists = {}
    print("No GCIM distribution data found — run nb 031 first.")

## Stage 1 - Build final ensembles (slow compute)

Runs the multi-round engine with `SELECTION_CONFIG_OPTIMISED` (see the table at the top) on the
**filtered** database and saves `AvgSA_03_final_ensembles.pickle` (+ a `.manifest.json`) in the
optimised folder. `db_exclusions` is passed to the engine so the stage caches and the final
manifest record the filter. Each step is provenance-cached: if an input changed since a cache
was written, the stage is recomputed; set `FORCE_RECOMPUTE = True` to rebuild.

Check in the printout: round 1 should read `Bounds  [bounded: m, vs30 | unbounded: d]` and
its optimisation phase should cover **all** stripes in the batch (`[force: ...]`).

Post-processing lives in **`033-gm_selection_AvgSA_03_stage2_postprocess-optimised.ipynb`**.

In [ ]:
# Force override: reselect every wanted stripe (ignores the per-stripe manifests).
if FORCE_RECOMPUTE:
    to_compute = list(wanted)

# Capture the prior COMPLETE final_ensembles before build_final_ensembles overwrites
# the file with just this batch. On a forced full rebuild we start clean.
prior_final = {}
if final_ensembles_fp.is_file() and not FORCE_RECOMPUTE:
    with open(final_ensembles_fp, "rb") as f:
        prior_final = pickle.load(f)

# Subset to the stale batch and run the multi-round engine over just those keys. The
# round loop is per-(site, iml) independent, so selecting a subset is equivalent to
# selecting them within the full set. gcim for every batch key must be present (built
# by nb 031); a missing key means 031 was not run for this batch.
#
# The disagg shards are loaded HERE, and only for the sites with stale stripes — an
# all-valid run reads no disagg data at all. The `gm_db` this call returns is the
# UNFILTERED database and is deliberately discarded (`*_`): only the filtered `gm_db`
# from the set-up cell is ever passed on.
if to_compute:
    stale_sites = {s for s, _ in to_compute}
    site_iml_disaggs, *_ = setup_AvgSA03_gcim_gm_selection(sites=stale_sites)
    batch = {k: site_iml_disaggs[k] for k in to_compute}
else:
    batch = {}

missing = [k for k in batch if k not in gcim_dists]
if missing:
    raise KeyError(
        f"{len(missing)} stale (site, iml) have no gcim — run nb 031 first. "
        f"e.g. {missing[:5]}")

if batch:
    batch_final = build_final_ensembles(
        batch,
        disagg_stats,
        gcim_dists,
        gm_db,                      # FILTERED database (set-up cell)
        basic_selection_ctx,
        site_model,
        source_fps=source_fps,
        stage_fps=stage_fps,
        output_fp=final_ensembles_fp,
        round_unbounded=SELECTION_CONFIG_OPTIMISED["round_unbounded"],
        force_optimisation=SELECTION_CONFIG_OPTIMISED["force_optimisation"],
        shuffle=SELECTION_CONFIG_OPTIMISED["shuffle"],
        n_shuffles=SELECTION_CONFIG_OPTIMISED["n_shuffles"],
        shuffle_rng_seeds=SELECTION_CONFIG_OPTIMISED["shuffle_rng_seeds"],
        rng_seed=rng_seed,
        force_recompute=FORCE_RECOMPUTE,
        # records the in-memory filter in the stage + final manifests (does not filter)
        db_exclusions=SELECTION_CONFIG_OPTIMISED["db_exclusions"],
    )
else:
    batch_final = {}
    print("All stripes valid — nothing to (re)select.")

# MERGE the batch into the prior complete set so final_ensembles.pickle stays the
# COMPLETE (site, iml) selection (read as complete by nb 040-optimised). Prune any key
# no longer wanted (an IML removed from a union list).
wanted_set = set(wanted)
final_ensembles = {k: v for k, v in {**prior_final, **batch_final}.items() if k in wanted_set}
if batch_final or prior_final:
    with open(final_ensembles_fp, "wb") as f:
        pickle.dump(final_ensembles, f)
print(f"final_ensembles.pickle now holds {len(final_ensembles)} (site, iml) "
      f"({len(batch_final)} (re)selected this run).")

In [ ]:
# Isolate the (site, iml) that still fail the KS test after all rounds (across the full
# merged set), mapping each to its list of failing IMs. Not pickled - just a quick look.
failing_ensembles = {
    k: v["ks_failed_ims"]
    for k, v in final_ensembles.items()
    if v is not None and not v["ks_passed"]
}

print(f"{len(failing_ensembles)} (site, iml) still failing:")
for k in failing_ensembles:
    print("  ", k, "->", failing_ensembles[k])

## Check: no excluded record was selected

Every selected record is identified by its row label in the database. Since the filter keeps
the original labels, an excluded record would show up here as one of the dropped labels.

In [ ]:
# Look for any dropped row label in every ensemble of the (merged, complete) set.
leaks = {
    k: sorted(set(v["recs"].index).intersection(excluded_idx))
    for k, v in final_ensembles.items()
    if v is not None
}
leaks = {k: v for k, v in leaks.items() if v}
if leaks:
    raise RuntimeError(f"Excluded records were selected in {len(leaks)} stripe(s): {leaks}")
print(f"OK - none of the {len(excluded_idx)} excluded database rows appears in any of the "
      f"{sum(v is not None for v in final_ensembles.values())} ensembles.")